# 04 — Parsing des PDF PTR House → transactions

**Rôle :** transformer le texte des PDF PTR en lignes de transactions. On porte en Python l'approche **ancrée sur marqueurs** de `seralifatih` (MIT).

**Fiabilité :** `ticker`, `asset_type`, `operation_type`, dates et fourchette de montant sont extraits de façon **fiable** par la regex. `asset_description` et `owner` sont *best-effort*. Le `disclosure_date` canonique vient de l'**index XML** (notebook 02), pas du PDF.

**Format réel d'une ligne** (vérifié) : `… Asset (TICKER) [TYPE]  P|S|S (partial)|E  date  notif_date  $low - $high`.

**Cellule 1 — Imports & chemins.**

In [2]:
import re
from pathlib import Path
import pandas as pd

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() or (p / ".git").exists():
            return p
    return start

PROJECT_ROOT  = find_project_root(Path.cwd())
PROC_HOUSE    = PROJECT_ROOT / "data" / "processed" / "house"
RAW_HOUSE_PDF = PROJECT_ROOT / "data" / "raw" / "house" / "ptr_pdfs"
ptr_index = pd.read_csv(PROC_HOUSE / "house_ptr_index.csv", dtype={"doc_id": str})
print("Entrées d'index :", len(ptr_index))

Entrées d'index : 8248


**Cellule 2 — Nettoyage du texte.** On retire les octets nuls et on aplatit les espaces (les PDF House collent / coupent les champs).

In [3]:
def normalize(text: str) -> str:
    t = (text or "").replace("\x00", " ")
    return re.sub(r"\s+", " ", t).strip()

**Cellule 3 — La regex ancrée sur marqueurs** (portée de seralifatih). Elle capture ticker, type d'actif, type d'opération, les deux dates et la fourchette.

In [4]:
TX = re.compile(
    r"\((?P<ticker>[A-Z][A-Z0-9.\-]{0,6})\)\s*"           # (TICKER)
    r"\[(?P<atype>[A-Z]{1,3})\]\s*"                         # [TYPE actif]
    r"(?P<ttype>[Ss]\s*\([Pp]artial\)|[Ss]|[Pp]|[Ee])\s+"  # opération (casse souple)
    r"(?P<tdate>\d{1,2}/\d{1,2}/\d{2,4})\s+"               # date transaction
    r"(?P<ndate>\d{1,2}/\d{1,2}/\d{2,4})\s+"               # date notification
    r"\$(?P<low>[\d,]+)\s*-\s*(?:\w+\s*)?\$(?P<high>[\d,]+)"  # fourchette (gfedc toléré)
)

# Format ancien (pre-2019) : (TICKER) sans [TYPE], casse souple, ticker peut contenir $
TX_OLD = re.compile(
    r"\((?P<ticker>[A-Z][A-Za-z0-9.$\-]{0,6})\)\s*"        # (ticker) casse souple + $ autorisé
    r"(?P<ttype>[Ss]\s*\([Pp]artial\)|[Ss]|[Pp]|[Ee])\s+"  # opération (casse souple)
    r"(?P<tdate>\d{1,2}/\d{1,2}/\d{2,4})\s+"               # date transaction
    r"(?P<ndate>\d{1,2}/\d{1,2}/\d{2,4})\s+"               # date notification
    r"\$(?P<low>[\d,]+)\s*-\s*(?:\w+\s*)?\$(?P<high>[\d,]+)"  # fourchette
)


**Cellule 4 — Parser le texte d'un PTR en lignes.** Pour chaque marqueur, le nom d'actif est pris dans l'écart précédent (après le dernier code propriétaire SP/JT/DC).

In [5]:
def parse_ptr(text: str) -> list:
    t = normalize(text)
    rows, prev_end = [], 0

    def extract_rows(pattern, atype_default=None):
        nonlocal prev_end
        nonlocal rows
        _prev = 0
        _rows = []
        for m in pattern.finditer(t):
            gap = t[_prev:m.start()]
            owners = list(re.finditer(r"\b(SP|JT|DC)\b", gap))
            if owners:
                owner = owners[-1].group(1)
                asset = gap[owners[-1].end():]
            else:
                owner = "SELF"
                asset = gap.split("S: New")[-1]
            asset = re.sub(r"\s+", " ", asset).strip(" .-")[-120:]
            atype = m.groupdict().get("atype") or atype_default
            _rows.append({
                "owner": owner, "asset_description": asset,
                "ticker": m.group("ticker").upper(), "asset_type": atype,
                "operation_raw": re.sub(r"\s+", " ", m.group("ttype")),
                "transaction_date": m.group("tdate"), "notification_date": m.group("ndate"),
                "amount_low": m.group("low"), "amount_high": m.group("high"),
            })
            _prev = m.end()
        return _rows

    # Format A : (TICKER) [TYPE] — format actuel
    rows_a = extract_rows(TX)
    if rows_a:
        return rows_a

    # Format B : (TICKER) sans [TYPE] — ancien format (pre-2019)
    rows_b = extract_rows(TX_OLD, atype_default=None)
    return rows_b


**Cellule 5 — Extraction du texte d'un PDF** (pdfplumber).

In [6]:
import pdfplumber
def extract_text(path: Path) -> str:
    try:
        with pdfplumber.open(path) as pdf:
            return "\n".join((p.extract_text() or "") for p in pdf.pages)
    except Exception:
        return "" 

**Cellule 6 — Boucle sur tous les PTR.** On parse chaque PDF présent et on rattache les métadonnées de l'index (déclarant, `disclosure_date = FilingDate`).

In [7]:
all_rows, unparsed = [], []
for _, r in ptr_index.iterrows():
    path = RAW_HOUSE_PDF / str(int(r["year"])) / f'{r["doc_id"]}.pdf'
    if not path.exists():
        continue
    parsed = parse_ptr(extract_text(path))
    if not parsed:
        unparsed.append(r["doc_id"]); continue
    for row in parsed:
        row.update({
            "source": "house", "provenance": "official",
            "declarant_name": f'{r.get("first_name","")} {r.get("last_name","")}'.strip(),
            "state_district": r.get("state_dst"),
            "disclosure_date": r.get("filing_date"),   # date publique canonique (XML)
            "doc_id": r["doc_id"], "source_url": r.get("pdf_url"),
        })
    all_rows.extend(parsed)
print("Transactions parsées :", len(all_rows), "| PDF non parsés :", len(unparsed))

Transactions parsées : 17053 | PDF non parsés : 5117


**Cellule 7 — Montants → midpoint + libellé d'opération.**

In [8]:
house_tx = pd.DataFrame(all_rows)
def to_int(s):
    return int(str(s).replace(",", "")) if pd.notna(s) and str(s) != "" else None
if len(house_tx):
    house_tx["amount_low"]  = house_tx["amount_low"].map(to_int)
    house_tx["amount_high"] = house_tx["amount_high"].map(to_int)
    house_tx["amount_range"] = house_tx.apply(
        lambda r: f"${r.amount_low:,} - ${r.amount_high:,}" if r.amount_low is not None else None, axis=1)
    house_tx["amount_midpoint"] = (house_tx["amount_low"] + house_tx["amount_high"]) / 2
    # Normalisation casse souple + "Sale (Partial)" avec P majuscule
    OP = {
        "P": "Purchase", "p": "Purchase",
        "S": "Sale",     "s": "Sale",
        "S (partial)": "Sale (Partial)", "s (partial)": "Sale (Partial)",
        "E": "Exchange", "e": "Exchange",
    }
    house_tx["operation_type"] = house_tx["operation_raw"].map(lambda x: OP.get(x, x))
house_tx.head(3)

,owner,asset_description,ticker,asset_type,operation_raw,transaction_date,notification_date,amount_low,amount_high,source,provenance,declarant_name,state_district,disclosure_date,doc_id,source_url,amount_range,amount_midpoint,operation_type
0,DC,ate/District: NJ01 tranSactionS iD owner asset...,HIL,NaN,s,12/26/2013,12/30/2013,15001,50000,house,official,Robert E. Andrews,NJ01,1/13/2014,20000077,https://disclosures-clerk.house.gov/public_dis...,"$15,001 - $50,000",32500.5,Sale
1,JT,"Dunkin' Brands group, Inc",DNKN,NaN,s,05/22/2014,05/22/2014,1001,15000,house,official,Lou Barletta,PA11,5/22/2014,20000998,https://disclosures-clerk.house.gov/public_dis...,"$1,001 - $15,000",8000.5,Sale
2,JT,Apple Inc,AAPL,NaN,S,10/1/2014,10/1/2014,15001,50000,house,official,Lou Barletta,PA11,10/3/2014,20001787,https://disclosures-clerk.house.gov/public_dis...,"$15,001 - $50,000",32500.5,Sale


**Cellule 8 — Sauvegarde + taux de parsing** (le reste va au backlog).

In [9]:
out = PROC_HOUSE / "house_transactions.csv"
house_tx.to_csv(out, index=False)
n_pdf = sum(1 for _, r in ptr_index.iterrows()
            if (RAW_HOUSE_PDF / str(int(r["year"])) / f'{r["doc_id"]}.pdf').exists())
print(f"Écrit : {out}")
print(f"PDF présents : {n_pdf} | parsés : {n_pdf - len(unparsed)} | backlog : {len(unparsed)}")

Écrit : /Users/lemairealice/Downloads/Jupiter/semaine 1/data/processed/house/house_transactions.csv
PDF présents : 8248 | parsés : 3131 | backlog : 5117


Transactions House extraites ✅ — passez au bloc Sénat **`05_senate_openset`**.